In [4]:
import numpy as np
import mne
from scipy.io import loadmat, savemat
from scipy.signal import hilbert
import matplotlib.pyplot as plt
from MFDFA import MFDFA # Assuming MFDFA library is available
import glob
import os

# 1. Load and Preprocess Data
def load_bci_data(file_path):
    """Load a single .mat file and return BCI structure."""
    mat = loadmat(file_path)
    bci = mat['BCI']
    # MATLAB .mat files often load struct arrays as (1,1) numpy object arrays
    if isinstance(bci, np.ndarray) and bci.shape == (1, 1):
        bci = bci[0, 0]
    return bci

# (create_mne_raw, plot_time_series, plot_psd, plot_erd_ers, plot_topography, and perform_mfdfa
# are unchanged from the previous version, focusing the change on PVC plotting)

def create_mne_raw(bci, trial_idx=4):
    """Create MNE Raw object for a single trial with filtering and bad channel interpolation."""
    # Get channel names
    try:
        if hasattr(bci, 'dtype') and bci.dtype.names and 'chaninfo' in bci.dtype.names:
            chaninfo = bci['chaninfo']
            if chaninfo.dtype.names and 'label' in chaninfo.dtype.names:
                label_field = chaninfo[0, 0]['label']
                # Flatten the array of arrays and extract string labels
                ch_names = [label.item() if isinstance(label, np.ndarray) else label for label in label_field.flatten()]
                if not all(isinstance(name, str) for name in ch_names):
                    # Attempt to convert any non-string labels to string
                    ch_names = [str(name) for name in ch_names]
            else:
                raise KeyError("label field not found in chaninfo sub-structure.")
        else:
            raise KeyError("chaninfo field not found in BCI structure.")
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing chaninfo.label: {e}")
        print("BCI type:", type(bci))
        print("BCI dtype.names:", bci.dtype.names if hasattr(bci, 'dtype') else "N/A")
        if hasattr(bci, 'dtype') and 'chaninfo' in bci.dtype.names:
            print("chaninfo content (partial):", bci['chaninfo'][0, 0] if bci['chaninfo'].shape == (1,1) else bci['chaninfo'])
            if bci['chaninfo'].dtype.names:
                print("chaninfo dtype.names:", bci['chaninfo'].dtype.names)
        raise

    # Get EEG data for the specified trial
    try:
        # Access the specific trial's EEG data
        eeg_data_trial = bci['data'][0, trial_idx]

        # Handle cases where eeg_data_trial might be 1D and needs reshaping
        if eeg_data_trial.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data_trial.size // n_channels
            if n_samples * n_channels == eeg_data_trial.size:
                eeg_data = eeg_data_trial.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data_trial.size} into (n_channels={n_channels}, n_samples). Sizes do not match.")
        elif eeg_data_trial.ndim == 3 and eeg_data_trial.shape[0] == 1:
            # Handle cases like (1, N_channels, N_samples)
            eeg_data = eeg_data_trial[0]
        elif eeg_data_trial.ndim == 2:
            eeg_data = eeg_data_trial
        else:
            raise ValueError(f"Unexpected EEG data dimension: {eeg_data_trial.ndim}. Expected 1, 2, or 3.")

        if eeg_data.shape[0] != len(ch_names):
            raise ValueError(f"EEG data's first dimension ({eeg_data.shape[0]}) does not match number of channels ({len(ch_names)}).")

        # Check for NaN/Inf in raw EEG data and replace with zeros
        if np.isnan(eeg_data).any():
            print(f"Warning: NaN values found in EEG data for trial {trial_idx} after extraction/reshaping. Replacing with zeros.")
            eeg_data = np.nan_to_num(eeg_data, nan=0.0)
        if np.isinf(eeg_data).any():
            print(f"Warning: Inf values found in EEG data for trial {trial_idx} after extraction/reshaping. Replacing with zeros.")
            eeg_data = np.nan_to_num(eeg_data, posinf=0.0, neginf=0.0)

    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing or processing EEG data for trial {trial_idx}: {e}")
        print("bci['data'] overall shape:", bci['data'].shape)
        if trial_idx < bci['data'].shape[1]:
            print(f"bci['data'][0, {trial_idx}] shape:", bci['data'][0, trial_idx].shape)
        raise

    sfreq = bci['SRATE'][0, 0]  # Sampling rate

    # Create MNE Info object
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

    # Create Raw object (convert data from microvolts to Volts for MNE)
    raw = mne.io.RawArray(eeg_data * 1e-6, info, verbose=False)

    # Set montage for channel locations. Use on_missing='ignore' to prevent error
    raw.set_montage('standard_1020', on_missing='ignore', verbose=False)

    # Exclude noisy channels (if present in data and MNE raw object)
    try:
        if 'chaninfo' in bci.dtype.names and 'noisechan' in bci['chaninfo'][0, 0].dtype.names:
            noisy_indices = bci['chaninfo'][0, 0]['noisechan']
            if noisy_indices is not None and noisy_indices.size > 0: # Check if array is not empty
                # Convert to 0-based indexing if necessary (MATLAB is 1-based)
                noisy_indices_0_based = noisy_indices.flatten() - 1

                bad_channels_from_bci = []
                for idx in noisy_indices_0_based:
                    if 0 <= idx < len(ch_names):
                        bad_channels_from_bci.append(ch_names[idx])
                    else:
                        print(f"Warning: Noisy channel index {idx+1} (MATLAB-based) is out of bounds for {len(ch_names)} channels.")

                if bad_channels_from_bci:
                    raw.info['bads'] = list(set(raw.info['bads'] + bad_channels_from_bci)) # Add to existing bads

                    # --- NEW FIX: Validate bad channels have valid positions before interpolation ---
                    valid_bads_to_interpolate = []
                    for bad_ch_name in raw.info['bads']:
                        try:
                            ch_idx = raw.info['ch_names'].index(bad_ch_name)
                            ch_info = raw.info['chs'][ch_idx]

                            # Check if location exists and contains no NaNs/Infs
                            if ch_info['loc'] is not None and \
                               not np.isnan(ch_info['loc']).any() and \
                               not np.isinf(ch_info['loc']).any():
                                valid_bads_to_interpolate.append(bad_ch_name)
                            else:
                                print(f"Warning: Bad channel '{bad_ch_name}' has invalid (None/NaN/Inf) location data. Skipping interpolation for this channel.")
                        except ValueError:
                            print(f"Warning: Bad channel '{bad_ch_name}' not found in raw.info['ch_names']. Skipping interpolation for this channel.")

                    # Update bads list to only include channels that can be interpolated
                    raw.info['bads'] = valid_bads_to_interpolate

                    if raw.info['bads']: # Only interpolate if there are valid bad channels left
                        # Ensure no NaNs/Infs BEFORE interpolation on raw._data itself, just in case
                        if np.isnan(raw.get_data()).any() or np.isinf(raw.get_data()).any():
                            print("NaN/Inf detected in raw data before bad channel interpolation. Replacing with zeros.")
                            raw._data = np.nan_to_num(raw.get_data(), nan=0.0, posinf=0.0, neginf=0.0)

                        raw.interpolate_bads(verbose=False)
                        print(f"Interpolated valid bad channels: {raw.info['bads']}")
                    else:
                        print("No valid bad channels with proper location data to interpolate.")
                else:
                    print("No noisy channels listed in 'noisechan' field or no valid bad channels identified.")
            else:
                print("No noisy channels listed in 'noisechan' field.")
        else:
            print("'noisechan' field not found in chaninfo.")
    except (IndexError, KeyError, TypeError) as e:
        print(f"Error processing noisy channels: {e}")
        print("Skipping noisy channel interpolation.")

    # Filter data: 1 Hz high-pass then 8-14 Hz bandpass (as per paper's make_ERD_figures.m)
    try:
        # Check for NaN/Inf immediately before filtering
        if np.isnan(raw.get_data()).any() or np.isinf(raw.get_data()).any():
            print("NaN/Inf detected in raw data before first filter pass. Attempting to clean.")
            raw._data = np.nan_to_num(raw.get_data(), nan=0.0, posinf=0.0, neginf=0.0)

        raw.filter(l_freq=1, h_freq=None, fir_design='firwin', verbose=False) # High-pass 1 Hz

        # Check for NaN/Inf after first filter pass
        if np.isnan(raw.get_data()).any() or np.isinf(raw.get_data()).any():
            print("NaN/Inf detected in raw data after 1Hz high-pass filter. Attempting to clean.")
            raw._data = np.nan_to_num(raw.get_data(), nan=0.0, posinf=0.0, neginf=0.0)

        raw.filter(l_freq=8, h_freq=14, fir_design='firwin', verbose=False) # Bandpass 8-14 Hz
    except ValueError as e:
        print(f"Error during raw.filter(): {e}. This might be due to NaNs/Infs introduced by a previous step or extreme data values.")
        # Attempt a robust fix if filter fails: force NaN/Inf to 0 and re-attempt filter
        print("Attempting to re-filter after robust NaN/Inf replacement.")
        raw._data = np.nan_to_num(raw.get_data(), nan=0.0, posinf=0.0, neginf=0.0)
        try:
            raw.filter(l_freq=1, h_freq=None, fir_design='firwin', verbose=False)
            raw.filter(l_freq=8, h_freq=14, fir_design='firwin', verbose=False)
        except Exception as retry_e:
            print(f"Re-filtering failed even after robust replacement: {retry_e}. Data remains problematic.")
            raise # Re-raise if even the robust attempt fails

    # Final check for NaN/Inf after all processing
    if np.isnan(raw.get_data()).any():
        print(f"Warning: NaN values found in EEG data for trial {trial_idx} after filtering. Replacing with zeros.")
        raw._data = np.nan_to_num(raw.get_data(), nan=0.0)
    if np.isinf(raw.get_data()).any():
        print(f"Warning: Inf values found in EEG data for trial {trial_idx} after filtering. Replacing with zeros.")
        raw._data = np.nan_to_num(raw.get_data(), posinf=0.0, neginf=0.0)

    return raw

# 2. Visualizations
def plot_time_series(raw, channels=['C3', 'C4']):
    """Plot EEG time series for selected channels."""
    # Ensure selected channels exist in raw.info['ch_names']
    available_channels = [ch for ch in channels if ch in raw.info['ch_names']]
    if not available_channels:
        print(f"None of the requested channels {channels} found in raw data for time series plot.")
        return
    try:
        # Create a copy of raw and pick only desired channels to avoid modifying original raw
        raw_to_plot = raw.copy().pick_channels(available_channels)
        raw_to_plot.plot(scalings='auto', n_channels=len(available_channels), show=False, block=False)
        plt.title('EEG Time Series (Selected Channels)')
        plt.tight_layout()
        plt.show() # Display plot
        plt.close()
    except ValueError as e:
        print(f"Error plotting time series: {e}. Check if channels {available_channels} exist and are valid for plotting.")

def plot_psd(raw, channels=['C3', 'C4']):
    """Plot PSD for selected channels."""
    available_channels = [ch for ch in channels if ch in raw.info['ch_names']]
    if not available_channels:
        print(f"None of the requested channels {channels} found in raw data for PSD plot.")
        return
    try:
        if raw.get_data().size == 0:
            print("Raw data is empty, cannot compute PSD.")
            return

        fig = raw.compute_psd(fmin=8, fmax=14, picks=available_channels, verbose=False).plot(show=False)
        fig.suptitle('Power Spectral Density (8-14 Hz)', y=0.98) # Adjust title position
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
        plt.show() # Display plot
        plt.close(fig) # Close the figure explicitly
    except ValueError as e:
        print(f"Error plotting PSD: {e}. Check if channels {available_channels} exist and data is valid.")
    except Exception as e:
        print(f"An unexpected error occurred during PSD plotting: {e}")


def plot_erd_ers(bci, trial_idx=4):
    """Plot ERD/ERS for C3/C4 during a trial using Hilbert transform and relative power change."""
    try:
        label_field = bci['chaninfo'][0, 0]['label']
        ch_names = [label.item() if isinstance(label, np.ndarray) else label for label in label_field.flatten()]

        # Ensure C3 and C4 exist
        if 'C3' not in ch_names or 'C4' not in ch_names:
            print("C3 or C4 channel not found for ERD/ERS plotting.")
            return
        c3_idx = ch_names.index('C3')
        c4_idx = ch_names.index('C4')

        eeg_data_trial = bci['data'][0, trial_idx]

        # Handle reshaping of eeg_data_trial
        if eeg_data_trial.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data_trial.size // n_channels
            eeg_data = eeg_data_trial.reshape(n_channels, n_samples)
        elif eeg_data_trial.ndim == 3 and eeg_data_trial.shape[0] == 1:
            eeg_data = eeg_data_trial[0]
        else:
            eeg_data = eeg_data_trial

        if eeg_data.shape[0] < max(c3_idx, c4_idx) + 1:
             print(f"Channel indices C3({c3_idx}) or C4({c4_idx}) out of bounds for EEG data shape {eeg_data.shape}")
             return

        # Check for trial artifact (from BCI.TrialData.artifact)
        try:
            is_artifact_trial = bci['TrialData'][0, trial_idx]['artifact'][0, 0]
            if is_artifact_trial:
                print(f"Trial {trial_idx+1} is marked as an artifact. Skipping ERD/ERS plot.")
                return
        except (KeyError, IndexError):
            print(f"Warning: 'artifact' field not found or malformed in TrialData for trial {trial_idx+1}. Proceeding without artifact check.")

        eeg_c3 = eeg_data[c3_idx, :] # Data in microvolts
        eeg_c4 = eeg_data[c4_idx, :] # Data in microvolts

    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in plot_erd_ers (data access): {e}")
        return

    sfreq = bci['SRATE'][0, 0]

    # Filter EEG data (1 Hz high-pass, 8-14 Hz bandpass) for Hilbert transform
    # Create a temporary MNE Raw object for filtering, then get data back in microvolts
    info_temp = mne.create_info(ch_names=['temp_c3', 'temp_c4'], sfreq=sfreq, ch_types='eeg', verbose=False)
    raw_temp = mne.io.RawArray(np.vstack((eeg_c3, eeg_c4)) * 1e-6, info_temp, verbose=False)
    raw_temp.filter(l_freq=1, h_freq=None, fir_design='firwin', verbose=False) # High-pass 1 Hz
    raw_temp.filter(l_freq=8, h_freq=14, fir_design='firwin', verbose=False) # Bandpass 8-14 Hz

    # Get filtered data back in microvolts
    filtered_c3 = raw_temp.get_data(picks='temp_c3').flatten() * 1e6
    filtered_c4 = raw_temp.get_data(picks='temp_c4').flatten() * 1e6

    # Apply Hilbert transform to get instantaneous power envelope
    hilbert_c3 = np.abs(hilbert(filtered_c3))
    hilbert_c4 = np.abs(hilbert(filtered_c4))

    # Define time windows based on paper (relative to an event/trial start)
    # The paper's MATLAB code for ERD_base uses BCI.time{1} for time indexing.
    # Assuming bci['time'][0,0] is the corresponding time vector in seconds.
    time_axis = bci['time'][0,0].flatten() # Ensure 1D array of time points

    # Find sample indices for baseline (-1000ms to 0ms relative to event)
    baseline_start_time_ms = -1000
    baseline_end_time_ms = 0

    # Convert milliseconds to seconds for comparison with time_axis
    baseline_start_sec = baseline_start_time_ms / 1000.0
    baseline_end_sec = baseline_end_time_ms / 1000.0

    # Find indices that match the time range
    baseline_indices = np.where((time_axis >= baseline_start_sec) & (time_axis <= baseline_end_sec))[0]

    # Feedback period (4000ms to resultind)
    feedback_start_time_ms = 4000
    feedback_start_sec = feedback_start_time_ms / 1000.0

    try:
        # resultind is the end of the trial in samples (from MATLAB code, it's a sample index)
        resultind_sample = int(bci['TrialData'][0, trial_idx]['resultind'][0, 0])
    except (IndexError, KeyError):
        print(f"Error accessing TrialData.resultind for trial {trial_idx}. Cannot plot ERD/ERS.")
        return

    # Ensure segments are not empty before computing mean power
    if len(baseline_indices) == 0:
        print(f"No baseline data found for trial {trial_idx + 1}. Check time axis or baseline definition.")
        return

    # Extract segments
    hilbert_c3_baseline = hilbert_c3[baseline_indices]
    hilbert_c4_baseline = hilbert_c4[baseline_indices]

    # Feedback samples: from 4s (converted to sample index) up to resultind
    feedback_start_sample_idx = np.where(time_axis >= feedback_start_sec)[0]
    if len(feedback_start_sample_idx) == 0:
        print(f"No feedback start time found for trial {trial_idx+1}. Check time axis or feedback definition.")
        return
    feedback_start_sample_idx = feedback_start_sample_idx[0]

    feedback_end_sample_idx = min(resultind_sample, len(hilbert_c3) - 1) # Ensure within bounds

    # Check if feedback segment is valid
    if feedback_start_sample_idx >= feedback_end_sample_idx:
        print(f"Feedback period for trial {trial_idx+1} is invalid (start: {feedback_start_sample_idx}, end: {feedback_end_sample_idx}). Skipping ERD/ERS.")
        return

    hilbert_c3_feedback = hilbert_c3[feedback_start_sample_idx:feedback_end_sample_idx]
    hilbert_c4_feedback = hilbert_c4[feedback_start_sample_idx:feedback_end_sample_idx]

    # Ensure segments are not empty
    if hilbert_c3_baseline.size == 0 or hilbert_c3_feedback.size == 0 or \
       hilbert_c4_baseline.size == 0 or hilbert_c4_feedback.size == 0:
        print(f"One or more EEG segments for ERD/ERS are empty for trial {trial_idx + 1}. Skipping plot.")
        return

    # Compute mean power in the specified segments
    mean_power_c3_baseline = np.nanmean(hilbert_c3_baseline)
    mean_power_c3_feedback = np.nanmean(hilbert_c3_feedback)
    mean_power_c4_baseline = np.nanmean(hilbert_c4_baseline)
    mean_power_c4_feedback = np.nanmean(hilbert_c4_feedback)

    # Compute ERD: ((feedback power - baseline power) / baseline power) * 100
    erd_c3 = ((mean_power_c3_feedback - mean_power_c3_baseline) / mean_power_c3_baseline) * 100 if mean_power_c3_baseline != 0 else np.nan
    erd_c4 = ((mean_power_c4_feedback - mean_power_c4_baseline) / mean_power_c4_baseline) * 100 if mean_power_c4_baseline != 0 else np.nan

    # Plot
    plt.figure(figsize=(6, 4))
    plt.bar(['C3', 'C4'], [erd_c3, erd_c4], color=['#FF6F61', '#6B5B95']) # Changed colors
    plt.ylabel('Relative Power Change (ERD/ERS) (%)')
    plt.title(f'ERD/ERS for Trial {trial_idx + 1}')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show() # Display plot
    plt.close()

def plot_topography(bci, trial_idx=4):
    """Plot topographic map of alpha power using Hilbert transform and relative power change."""
    try:
        label_field = bci['chaninfo'][0, 0]['label']
        ch_names = [label.item() if isinstance(label, np.ndarray) else label for label in label_field.flatten()]
        eeg_data_trial = bci['data'][0, trial_idx] # Data in microvolts

        # Handle reshaping of eeg_data_trial
        if eeg_data_trial.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data_trial.size // n_channels
            eeg_data = eeg_data_trial.reshape(n_channels, n_samples)
        elif eeg_data_trial.ndim == 3 and eeg_data_trial.shape[0] == 1:
            eeg_data = eeg_data_trial[0]
        else:
            eeg_data = eeg_data_trial

        # Check for trial artifact
        try:
            is_artifact_trial = bci['TrialData'][0, trial_idx]['artifact'][0, 0]
            if is_artifact_trial:
                print(f"Trial {trial_idx+1} is marked as an artifact. Skipping topography plot.")
                return
        except (KeyError, IndexError):
            print(f"Warning: 'artifact' field not found or malformed in TrialData for trial {trial_idx+1}. Proceeding without artifact check.")

    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing EEG data for topography: {e}")
        return

    sfreq = bci['SRATE'][0, 0]

    # Create temporary MNE Raw object for filtering
    info_temp = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg', verbose=False)
    raw_temp = mne.io.RawArray(eeg_data * 1e-6, info_temp, verbose=False) # Convert to Volts for MNE
    raw_temp.set_montage('standard_1020', on_missing='ignore', verbose=False)

    # Filter data (1 Hz high-pass, 8-14 Hz bandpass)
    raw_temp.filter(l_freq=1, h_freq=None, fir_design='firwin', verbose=False)
    raw_temp.filter(l_freq=8, h_freq=14, fir_design='firwin', verbose=False)

    # Get filtered data back in microvolts
    filtered_eeg_data = raw_temp.get_data() * 1e6

    # Apply Hilbert transform to get instantaneous power envelope for all channels
    hilbert_eeg_data = np.abs(hilbert(filtered_eeg_data))

    # Define time windows based on paper (relative to an event/trial start)
    time_axis = bci['time'][0,0].flatten() # Ensure 1D array of time points

    # Find sample indices for baseline (-1000ms to 0ms relative to event)
    baseline_start_sec = -1000 / 1000.0
    baseline_end_sec = 0 / 1000.0
    baseline_indices = np.where((time_axis >= baseline_start_sec) & (time_axis <= baseline_end_sec))[0]

    # Feedback period (4000ms to resultind)
    feedback_start_sec = 4000 / 1000.0
    try:
        resultind_sample = int(bci['TrialData'][0, trial_idx]['resultind'][0, 0])
    except (IndexError, KeyError):
        print(f"Error accessing TrialData.resultind for trial {trial_idx}. Cannot plot topography.")
        return

    feedback_start_sample_idx = np.where(time_axis >= feedback_start_sec)[0]
    if len(feedback_start_sample_idx) == 0:
        print(f"No feedback start time found for trial {trial_idx+1}. Check time axis or feedback definition.")
        return
    feedback_start_sample_idx = feedback_start_sample_idx[0]

    feedback_end_sample_idx = min(resultind_sample, hilbert_eeg_data.shape[1] - 1)

    if len(baseline_indices) == 0:
        print(f"No baseline data found for topography for trial {trial_idx + 1}. Skipping plot.")
        return
    if feedback_start_sample_idx >= feedback_end_sample_idx:
        print(f"Feedback period for topography for trial {trial_idx+1} is invalid. Skipping plot.")
        return

    # Compute mean Hilbert power for baseline and feedback for each channel
    hilbert_base_power = np.nanmean(hilbert_eeg_data[:, baseline_indices], axis=1)
    hilbert_feed_power = np.nanmean(hilbert_eeg_data[:, feedback_start_sample_idx:feedback_end_sample_idx], axis=1)

    # Compute ERD_topo: ((feedback power - baseline power) / baseline power) * 100
    # Handle division by zero for channels with zero or near-zero baseline power
    erd_topo_values = np.zeros_like(hilbert_base_power)
    # Add a small epsilon to the denominator to prevent division by zero for very small baseline values
    epsilon = np.finfo(float).eps
    non_zero_baseline = np.abs(hilbert_base_power) > epsilon
    erd_topo_values[non_zero_baseline] = ((hilbert_feed_power[non_zero_baseline] - hilbert_base_power[non_zero_baseline]) / hilbert_base_power[non_zero_baseline]) * 100
    # For channels with effectively zero baseline, set ERD to NaN
    erd_topo_values[~non_zero_baseline] = np.nan

    # --- FIX for overlapping electrodes: Exclude problematic channels from plotting ---
    problematic_channels = ['FP1', 'FPZ', 'FP2', 'FZ', 'FCZ', 'CZ', 'CPZ', 'PZ', 'POZ', 'CB1', 'OZ', 'CB2']
    
    channels_to_plot_names = []
    erd_values_to_plot = []
    
    for idx, ch_name in enumerate(raw_temp.ch_names):
        if ch_name not in problematic_channels:
            # Ensure channel has valid position for plotting
            ch_info = raw_temp.info['chs'][idx]
            if ch_info['loc'] is not None and \
               not np.isnan(ch_info['loc']).any() and \
               not np.isinf(ch_info['loc']).any():
                channels_to_plot_names.append(ch_name)
                erd_values_to_plot.append(erd_topo_values[idx])
            else:
                print(f"Warning: Channel '{ch_name}' has invalid location data or is problematic. Skipping for topography.")

    if not channels_to_plot_names:
        print("No valid channels with proper location data to plot topography after filtering problematic ones.")
        return

    try:
        # Create a new info object with only channels that have valid positions and are not problematic
        info_for_topomap = mne.create_info(
            ch_names=channels_to_plot_names,
            sfreq=raw_temp.info['sfreq'],
            ch_types='eeg',
            verbose=False
        )
        info_for_topomap.set_montage('standard_1020', on_missing='ignore', verbose=False)

        # Ensure that `erd_values_to_plot` contains no NaNs or Infs
        erd_values_to_plot = np.nan_to_num(np.array(erd_values_to_plot), nan=0.0, posinf=0.0, neginf=0.0)

        fig, ax = plt.subplots(figsize=(6, 6)) # Create a figure and axis for the plot
        im, cn = mne.viz.plot_topomap(erd_values_to_plot, info_for_topomap, axes=ax, show=False)
        plt.title(f'ERD Topography (Trial {trial_idx + 1})')
        # Add a colorbar
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
        cbar.set_label(r'Relative Power Change (%)') # Label for ERD topography
        plt.tight_layout()
        plt.show() # Display plot
        plt.close(fig) # Close the figure explicitly
    except Exception as e:
        print(f"An error occurred during topographic plot generation: {e}")


def compute_pvc(bci):
    """Compute Percent Valid Correct (PVC)."""
    try:
        # Check if TrialData exists and has content
        if 'TrialData' not in bci.dtype.names or bci['TrialData'].size == 0:
            # print("TrialData field is missing or empty, cannot compute PVC.") # Suppress for multi-subject
            return None

        results = []
        for i in range(bci['TrialData'].shape[1]):
            trial_data_element = bci['TrialData'][0, i]
            if 'result' in trial_data_element.dtype.names:
                result_val = trial_data_element['result'][0, 0]
                results.append(result_val)
            # else:
                # print(f"Warning: 'result' field not found for trial {i}. Skipping this trial for PVC.") # Suppress for multi-subject

        if not results:
            # print("No valid trial results found to compute PVC.") # Suppress for multi-subject
            return None

        results_array = np.array(results)
        hits = np.sum(results_array == 1)
        misses = np.sum(results_array == 0)

        return hits / (hits + misses) * 100 if (hits + misses) > 0 else 0
    except (IndexError, KeyError, TypeError) as e:
        # print(f"Error computing PVC: {e}") # Suppress for multi-subject
        return None

def plot_pvc_across_all_subjects(subject_ids, base_path='.'):
    """Plot PVC across sessions for multiple subjects in one graph."""
    plt.figure(figsize=(10, 6)) # Larger figure for multiple lines
    all_pvc_data = {} # To store PVC values for each subject

    max_sessions = 0

    for subject_id in subject_ids:
        pvc_list = []
        # Adjust glob to look for files in a specified base_path if needed
        # Assuming session files are named like 'S1_Session_1.mat', 'S1_Session_2.mat', etc.
        session_files = sorted(glob.glob(f'{base_path}/{subject_id}_Session_*.mat'))

        if not session_files:
            print(f"No session files found for {subject_id} in {base_path}. Skipping.")
            continue

        for file in session_files:
            # print(f"Processing {file} for PVC...") # Comment out for cleaner output
            try:
                bci = load_bci_data(file)
                pvc = compute_pvc(bci)
                if pvc is not None:
                    pvc_list.append(pvc)
                else:
                    pvc_list.append(np.nan) # Append NaN if PVC cannot be computed
            except Exception as e:
                print(f"Error loading or processing {file}: {e}. Appending NaN.")
                pvc_list.append(np.nan) # Append NaN if file cannot be loaded/processed

        if pvc_list:
            all_pvc_data[subject_id] = pvc_list
            if len(pvc_list) > max_sessions:
                max_sessions = len(pvc_list)

    if not all_pvc_data:
        print("No valid PVC data found for any subject to plot.")
        return

    # Plot each subject's PVC
    for subject_id, pvc_values in all_pvc_data.items():
        # Pad shorter lists with NaN to match max_sessions for consistent x-axis
        padded_pvc_values = pvc_values + [np.nan] * (max_sessions - len(pvc_values))
        plt.plot(range(1, len(padded_pvc_values) + 1), padded_pvc_values, marker='o', linestyle='-', label=subject_id, alpha=0.7)

    plt.xlabel('Session Number')
    plt.ylabel('Percent Valid Correct (%)')
    plt.title('PVC Across Sessions for All Subjects')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.xticks(range(1, max_sessions + 1)) # Ensure integer ticks for sessions
    plt.ylim(0, 100) # PVC is a percentage
    plt.legend(title='Subject ID', bbox_to_anchor=(1.05, 1), loc='upper left') # Move legend outside
    plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust layout to make space for the legend
    plt.show()
    plt.close()


# 3. MFDFA Analysis
def perform_mfdfa(bci, channel='C3', trial_idx=4):
    """Perform MFDFA on a single channel's EEG data, excluding artifacted trials."""
    try:
        label_field = bci['chaninfo'][0, 0]['label']
        ch_names = [label.item() if isinstance(label, np.ndarray) else label for label in label_field.flatten()]

        if channel not in ch_names:
            print(f"Channel '{channel}' not found in data for MFDFA.")
            return None, None, None

        ch_idx = ch_names.index(channel)
        eeg_data_trial = bci['data'][0, trial_idx]

        # Handle reshaping of eeg_data_trial
        if eeg_data_trial.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data_trial.size // n_channels
            eeg_data = eeg_data_trial.reshape(n_channels, n_samples)
        elif eeg_data_trial.ndim == 3 and eeg_data_trial.shape[0] == 1:
            eeg_data = eeg_data_trial[0]
        else:
            eeg_data = eeg_data_trial

        if eeg_data.shape[0] < ch_idx + 1:
            print(f"Channel index {ch_idx} out of bounds for EEG data shape {eeg_data.shape}")
            return None, None, None

        # Check for trial artifact
        try:
            is_artifact_trial = bci['TrialData'][0, trial_idx]['artifact'][0, 0]
            if is_artifact_trial:
                print(f"Trial {trial_idx+1} is marked as an artifact. Skipping MFDFA.")
                return None, None, None
        except (KeyError, IndexError):
            print(f"Warning: 'artifact' field not found or malformed in TrialData for trial {trial_idx+1}. Proceeding without artifact check.")

        eeg_data_channel = eeg_data[ch_idx, :] # Data in microvolts
    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in perform_mfdfa (data access): {e}")
        return None, None, None

    # Check for NaN/Inf in extracted channel data BEFORE MFDFA
    if np.isnan(eeg_data_channel).any() or np.isinf(eeg_data_channel).any():
        print("NaN/Inf detected in channel data before MFDFA. Replacing with zeros.")
        eeg_data_channel = np.nan_to_num(eeg_data_channel, nan=0.0, posinf=0.0, neginf=0.0)

    # MFDFA parameters
    data_length = len(eeg_data_channel)
    if data_length < 256: # Minimum length for meaningful MFDFA
        print(f"Warning: Data length ({data_length}) is too small for MFDFA. Skipping analysis.")
        return None, None, None

    # --- FIX for MFDFA error: Use powers of 2 for scales to ensure robustness ---
    min_log_scale = int(np.log2(16)) # Start from 16
    max_log_scale = int(np.log2(data_length // 4)) # Up to 1/4 of data length
    
    scales = np.array([2**i for i in range(min_log_scale, max_log_scale + 1)])
    scales = np.unique(scales) # Ensure unique scales
    
    # Ensure there are at least 2 scales for meaningful analysis
    if len(scales) < 2:
        print(f"Not enough distinct scales ({len(scales)}) for MFDFA due to data length ({data_length}). Skipping MFDFA.")
        return None, None, None
    
    q = np.linspace(-5, 5, 101)  # q-orders for multifractal analysis

    # Compute MFDFA
    try:
        lag, dfa = MFDFA(eeg_data_channel, scales, q)
    except ValueError as e:
        print(f"Error during MFDFA computation: {e}. This might be due to insufficient data length or invalid scales, or an internal MFDFA library issue.")
        return None, None, None
    except Exception as e:
        print(f"An unexpected error occurred during MFDFA computation: {e}")
        return None, None, None

    # Calculate Hurst exponent
    h_q = []
    # Ensure dfa has enough points for polyfit and that lag is also valid
    if lag.size < 2 or dfa.shape[1] < 2 or np.any(~np.isfinite(np.log2(lag))):
        print("Not enough data points or invalid lag for polyfit in MFDFA Hurst exponent calculation. Skipping singularity spectrum.")
        return None, None, None # Return None for alpha, f_alpha, h_q

    for i in range(len(q)):
        # Filter out NaN or inf values if any before fitting
        valid_indices = np.isfinite(np.log2(lag)) & np.isfinite(np.log2(dfa[i, :]))
        if np.sum(valid_indices) > 1: # Need at least 2 points for polyfit
            coef = np.polyfit(np.log2(lag[valid_indices]), np.log2(dfa[i, valid_indices]), 1)
            h_q.append(coef[0])
        else:
            h_q.append(np.nan) # Append NaN if fit is not possible
    h_q = np.array(h_q)

    # Singularity spectrum (only if h_q is valid)
    if not np.all(np.isnan(h_q)):
        valid_q_indices = np.isfinite(h_q) # Only use finite h_q values
        q_valid = q[valid_q_indices]
        h_q_valid = h_q[valid_q_indices]

        if len(q_valid) > 1:
            grad_h_q = np.gradient(h_q_valid, q_valid[1] - q_valid[0])
        else:
            grad_h_q = np.array([0.0]) # Handle as per context

        alpha = h_q_valid + q_valid * grad_h_q

        # Ensure dfa has enough columns for index -1. If not, pick a valid index or default
        # Add a check for dfa_last_valid to avoid issues if dfa is malformed
        if dfa.shape[1] > 0 and dfa[valid_q_indices, -1].size == len(q_valid):
            dfa_last_valid = dfa[valid_q_indices, -1]
        else:
            dfa_last_valid = np.ones_like(q_valid) * np.finfo(float).eps # Default to small positive value

        # Handle log of zero or negative if MFDFA returns problematic values
        with np.errstate(divide='ignore', invalid='ignore'): # Suppress warnings for log(0)
            f_alpha = q_valid * alpha - (h_q_valid * q_valid - np.log2(np.where(dfa_last_valid > 0, dfa_last_valid, np.finfo(float).eps)))

        # Remove NaNs or Infs that might arise from calculations
        valid_spectrum_indices = np.isfinite(alpha) & np.isfinite(f_alpha)
        alpha = alpha[valid_spectrum_indices]
        f_alpha = f_alpha[valid_spectrum_indices]

        if len(alpha) == 0:
            print("Singularity spectrum could not be computed (all NaNs/Infs after processing).")
            return None, None, None

        # Sort by alpha for plotting
        sort_indices = np.argsort(alpha)
        alpha = alpha[sort_indices]
        f_alpha = f_alpha[sort_indices]

        # Plot multifractal spectrum
        plt.figure(figsize=(6, 4))
        plt.plot(alpha, f_alpha, 'b-', linewidth=2)
        plt.xlabel(r'Singularity Strength ($\alpha$)') # LaTeX for alpha
        plt.ylabel(r'Multifractal Spectrum ($f(\alpha)$)') # LaTeX for f(alpha)
        plt.title(f'MFDFA Spectrum for {channel} (Trial {trial_idx + 1})')
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show() # Display plot
        plt.close()

        return alpha, f_alpha, h_q
    else:
        print("Hurst exponents (h_q) could not be reliably computed. Skipping singularity spectrum.")
        return None, None, None

# Main execution block
if __name__ == '__main__':
    # Define a default file path for a single subject's analysis example
    single_subject_file_path = 'S1_Session_1.mat'

    # --- Dummy .mat file creation for demonstration if file_path does not exist ---
    # This block will create dummy files for S1 to S20, each with multiple sessions.
    # It's crucial for demonstrating the multi-subject PVC plot without actual data.
    
    # Parameters for dummy data (tuned for demonstration, balance between data size and analysis)
    num_channels_dummy = 60
    sfreq_dummy = 1000 # Hz
    num_samples_per_trial_dummy = 2000 # Reduced from 12000 in original dummy, for faster processing
    num_trials_dummy = 5 # Number of trials per session
    max_sessions_per_subject = 5 # Number of sessions per subject

    subject_ids_for_dummy_creation = [f'S{i}' for i in range(1, 21)] # S1 to S20

    print("--- Creating Dummy .mat files for demonstration ---")
    for sub_id in subject_ids_for_dummy_creation:
        for session_num in range(1, max_sessions_per_subject + 1):
            file_name = f"{sub_id}_Session_{session_num}.mat"
            if not os.path.exists(file_name):
                # Create dummy EEG data (microvolts)
                dummy_eeg_data = np.random.rand(num_channels_dummy, num_samples_per_trial_dummy) * 200 - 100

                dummy_ch_names = [f'EEG {i:03d}' for i in range(1, num_channels_dummy + 1)]
                if 'C3' not in dummy_ch_names: dummy_ch_names[2] = 'C3'
                if 'C4' not in dummy_ch_names: dummy_ch_names[3] = 'C4'

                dummy_time_vector = np.linspace(-2, (num_samples_per_trial_dummy - 1) / sfreq_dummy - 2, num_samples_per_trial_dummy)

                dt_label = np.dtype([('label', 'O')])
                dt_noise = np.dtype([('noisechan', 'O')])
                dummy_chaninfo_labels = np.array([[np.array([name]) for name in dummy_ch_names]], dtype=dt_label)
                dummy_chaninfo_noise = np.array([[np.array([5])]], dtype=dt_noise)
                dummy_chaninfo_structured = np.zeros((1, 1), dtype={'names':['label', 'noisechan'],
                                                                    'formats':[dummy_chaninfo_labels.dtype, dummy_chaninfo_noise.dtype]})
                dummy_chaninfo_structured['label'][0,0] = dummy_chaninfo_labels[0,0]
                dummy_chaninfo_structured['noisechan'][0,0] = dummy_chaninfo_noise[0,0]

                dummy_trial_data_dtype = np.dtype([('result', 'O'), ('resultind', 'O'), ('tasknumber', 'O'), ('artifact', 'O')])
                dummy_trial_data_array = np.zeros((1, num_trials_dummy), dtype=dummy_trial_data_dtype)
                for i in range(num_trials_dummy):
                    dummy_trial_data_array[0, i]['result'] = np.array([[np.random.choice([0, 1])]]) # Random 0 or 1 for hit/miss
                    dummy_trial_data_array[0, i]['resultind'] = np.array([[num_samples_per_trial_dummy - 1]])
                    dummy_trial_data_array[0, i]['tasknumber'] = np.array([[1]])
                    dummy_trial_data_array[0, i]['artifact'] = np.array([[0]]) # Mostly no artifacts

                # Introduce some random artifacts
                if np.random.rand() < 0.2: # 20% chance of an artifacted trial
                    dummy_trial_data_array[0, np.random.randint(num_trials_dummy)]['artifact'] = np.array([[1]])

                dummy_bci = {
                    'data': np.array([[dummy_eeg_data] * num_trials_dummy], dtype=object),
                    'time': np.array([[dummy_time_vector]], dtype=object),
                    'SRATE': np.array([[sfreq_dummy]]),
                    'TrialData': dummy_trial_data_array,
                    'chaninfo': dummy_chaninfo_structured,
                    'positionx': np.array([[]], dtype=object),
                    'positiony': np.array([[]], dtype=object),
                    'metadata': np.array([[]], dtype=object),
                }
                savemat(file_name, {'BCI': dummy_bci})
                # print(f"Created dummy file: {file_name}") # Comment out for cleaner output
    print("--- Dummy .mat file creation complete ---")
    # --- End of dummy .mat file creation block ---


    try:
        # Example for a single subject (S1, Session 1) for other plots
        print(f"\n--- Running example analysis for {single_subject_file_path} ---")
        if os.path.exists(single_subject_file_path):
            bci_single_subject = load_bci_data(single_subject_file_path)

            # Determine a valid trial_idx for the single subject analysis
            trial_idx_single = 0
            num_trials_available_single = bci_single_subject['data'].shape[1]
            found_valid_trial_single = False
            for i in range(num_trials_available_single):
                try:
                    is_artifact_single = bci_single_subject['TrialData'][0, i]['artifact'][0, 0]
                    if not is_artifact_single:
                        trial_idx_single = i
                        found_valid_trial_single = True
                        print(f"Using non-artifact trial index {trial_idx_single+1} for single subject analysis.")
                        break
                except (KeyError, IndexError):
                    print(f"Warning: Could not check artifact status for trial {i+1} in {single_subject_file_path}. Assuming it's valid.")
                    trial_idx_single = i
                    found_valid_trial_single = True
                    break

            if not found_valid_trial_single:
                print(f"No non-artifact trials found or able to be checked for {single_subject_file_path}. Defaulting to trial 1.")
                trial_idx_single = 0
                if num_trials_available_single == 0:
                    print("No trials available in the BCI data for single subject analysis. Skipping.")
                else:
                    raw_single = create_mne_raw(bci_single_subject, trial_idx=trial_idx_single)
                    print("MNE Raw object created successfully for single subject.")

                    print("\n--- Generating Visualizations for single subject ---")
                    plot_time_series(raw_single, channels=['C3', 'C4'])
                    plot_psd(raw_single, channels=['C3', 'C4'])
                    plot_erd_ers(bci_single_subject, trial_idx=trial_idx_single)
                    plot_topography(bci_single_subject, trial_idx=trial_idx_single)

                    print("\n--- Performing MFDFA Analysis for single subject ---")
                    alpha, f_alpha, h_q = perform_mfdfa(bci_single_subject, channel='C3', trial_idx=trial_idx_single)
                    if alpha is not None:
                        print(f'Multifractal spectrum width (single subject): {np.max(alpha) - np.min(alpha):.2f}')
                    else:
                        print("MFDFA analysis skipped for single subject.")
            else:
                raw_single = create_mne_raw(bci_single_subject, trial_idx=trial_idx_single)
                print("MNE Raw object created successfully for single subject.")

                print("\n--- Generating Visualizations for single subject ---")
                plot_time_series(raw_single, channels=['C3', 'C4'])
                plot_psd(raw_single, channels=['C3', 'C4'])
                plot_erd_ers(bci_single_subject, trial_idx=trial_idx_single)
                plot_topography(bci_single_subject, trial_idx=trial_idx_single)

                print("\n--- Performing MFDFA Analysis for single subject ---")
                alpha, f_alpha, h_q = perform_mfdfa(bci_single_subject, channel='C3', trial_idx=trial_idx_single)
                if alpha is not None:
                    print(f'Multifractal spectrum width (single subject): {np.max(alpha) - np.min(alpha):.2f}')
                else:
                    print("MFDFA analysis skipped for single subject.")
        else:
            print(f"Single subject example file '{single_subject_file_path}' not found, skipping specific plots.")

        # --- Plot PVC for all subjects (S1-S20) ---
        print("\n--- Generating PVC plot for all subjects (S1-S20) ---")
        all_subject_ids = [f'S{i}' for i in range(1, 21)] # Subjects S1 to S20
        plot_pvc_across_all_subjects(all_subject_ids)

    except Exception as e:
        print(f"\nAn error occurred during execution: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for debugging

--- Creating Dummy .mat files for demonstration ---


KeyboardInterrupt: 